In [1]:
import pandas as pd

In [2]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

In [3]:
sql_data = [
    "Q: What is SQL? A: SQL is a structured query language used to manage and manipulate relational databases.",
    "Q: Difference between SQL and MySQL? A: SQL is a query language, while MySQL is a relational database management system that uses SQL.",
    "Q: What is a database? A: A database is an organized collection of structured data stored electronically.",
    "Q: What is a table? A: A table stores data in rows and columns.",
    "Q: What is a primary key? A: A primary key uniquely identifies each record in a table and cannot contain NULL values.",
    "Q: What is a foreign key? A: A foreign key links one table to another by referencing a primary key.",
    "Q: What is normalization? A: Normalization organizes data to reduce redundancy and improve integrity.",
    "Q: What is a JOIN? A: A JOIN combines rows from multiple tables based on related columns.",
    "Q: What is WHERE clause? A: WHERE filters rows before aggregation.",
    "Q: What is HAVING clause? A: HAVING filters aggregated results after GROUP BY.",
    "Q: Difference between WHERE and HAVING? A: WHERE filters before aggregation; HAVING filters after aggregation.",
    "Q: What is GROUP BY? A: GROUP BY groups rows with similar values for aggregation.",
    "Q: What is ORDER BY? A: ORDER BY sorts query results in ascending or descending order.",
    "Q: What are aggregate functions? A: Aggregate functions perform calculations such as SUM, COUNT, AVG, MIN, and MAX.",
    "Q: What is COUNT()? A: COUNT() returns the number of rows matching a condition.",
    "Q: What is an index? A: An index improves query performance by creating a fast lookup structure.",
    "Q: What are window functions? A: Window functions perform calculations across a set of rows related to the current row.",
    "Q: What is ROW_NUMBER()? A: ROW_NUMBER() assigns a unique sequential number to rows.",
    "Q: What is RANK()? A: RANK() assigns ranking values and skips duplicates.",
    "Q: What is LAG()? A: LAG() accesses data from a previous row in the result set.",
    "Q: What is LEAD()? A: LEAD() accesses data from a subsequent row in the result set.",
    "Q: What is a subquery? A: A subquery is a query nested inside another query.",
    "Q: What is a CTE? A: A Common Table Expression is a temporary result set defined within a query using WITH.",
]

In [4]:
print("Total Q&A: ", len(sql_data))

Total Q&A:  23


In [5]:
# Loading the Embedding Model from sentence transformers having the dimension of 384
model = SentenceTransformer("all-MiniLM-L6-v2")
print("Embedding model loaded successfully.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded successfully.


In [6]:
embedding = model.encode(sql_data)

In [7]:
print("Embedding Shape: ", embedding.shape)

Embedding Shape:  (23, 384)


In [8]:
embeddings = np.array(embedding).astype("float32")

In [9]:
print("Data Type: ", embeddings.dtype)

Data Type:  float32


In [10]:
faiss.normalize_L2(embeddings)
print("Embedding are Normalized")

Embedding are Normalized


In [11]:
# creating the FAISS index we are using IndexFlatIP which calculates the cosine-similarity between the vectors
dimension = embeddings.shape[1]  # 384
index = faiss.IndexFlatIP(dimension)
print("FAISS index created.")

FAISS index created.


In [12]:
index.add(embeddings)
print("Total vectors in FAISS index:", index.ntotal)

Total vectors in FAISS index: 23


In [13]:
# search function so that the query will be transformed into embedding, it will normalize it, search FAISS and give us top results

def search_sql(query, top_k=3):
    # Convert query to embedding
    query_embedding = model.encode([query])
    query_embedding = np.array(query_embedding).astype("float32")
    
    # Normalize (same as stored vectors)
    faiss.normalize_L2(query_embedding)
    
    # Search in FAISS
    scores, indices = index.search(query_embedding, top_k)
    
    print(f"\nQuery: {query}\n")
    print("Top matches:\n")
    
    for i, idx in enumerate(indices[0]):
        print(f"Match {i+1} (Score: {scores[0][i]:.4f})")
        print(sql_data[idx])
        print("-" * 50)

In [15]:
search_sql("What is the aggregation in SQL?")


Query: What is the aggregation in SQL?

Top matches:

Match 1 (Score: 0.6466)
Q: What are aggregate functions? A: Aggregate functions perform calculations such as SUM, COUNT, AVG, MIN, and MAX.
--------------------------------------------------
Match 2 (Score: 0.6407)
Q: What is GROUP BY? A: GROUP BY groups rows with similar values for aggregation.
--------------------------------------------------
Match 3 (Score: 0.6130)
Q: What is WHERE clause? A: WHERE filters rows before aggregation.
--------------------------------------------------
